[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q1/03_consensus_compare.ipynb)

# R1-Q1 Week 4 — Consensus Comparison & Report

You're in Week 4 of R1-Q1. Last week you identified the common stress core — 9,067 genes differentially expressed in at least 6 of the 8 abiotic stresses in AtGenExpress — and ran GO over-representation analysis to characterize what those genes do functionally. This week takes that core and compares it to a published meta-analytic consensus: Hakkak & Tohidfar 2026's signature of conserved drought and salt response in *Arabidopsis thaliana*. The comparison closes the R1-Q1 arc and produces the material for the final paper.

**By the end of this notebook you will have:**

- The core gene set translated from ATH1 probe IDs to AGI locus codes.
- A pre-committed prediction of how the two cores should overlap, written down before seeing the data.
- The overlap result — shared genes, ours-only, theirs-only — with hypergeometric significance, plus a three-region breakdown checked against the prediction.
- Targeted comparisons against Hakkak & Tohidfar's transcription factor list and the specific hub genes named in their paper.
- A synthesis pulling the comparison together with the enrichment findings from Notebook 2, and a saved comparison artifact for the final paper.

## 1) Setup

Install the library, import the scientific Python stack, mount Drive, and confirm Notebook 2's output is where this notebook expects to find it.

### 1.2) Installing the library

If you are running this notebook in Colab for the first time today, run the install cell below, then restart the runtime (Runtime → Restart session). After the restart, skip the install cell and run from the imports cell onward.

The runtime restart is what makes the freshly installed library version actually load. Without it, Python may continue using a cached older version and you will see confusing errors that look like missing functions even though pip reports a successful install.

If you have already restarted the runtime since installing v0.2.0 today, you can skip the install cell entirely.

In [ ]:
# Setup Step 1: Install the iRI Lab library
# 1. Install the library (not pre-installed on Colab).
# There are four varieties presented here, in order of decreasing complexity and time requirement. 
# You can choose any one, but you should only run one (not all four). 
# The first is the most aggressive in terms of forcing a reinstall, while the last 
# is the least aggressive. If you are not sure which one to use, you can start
# with the last one and see if it works. If it does not work, you can try the next 
# one, and so on. 

#!pip install --upgrade --force-reinstall --no-cache-dir git+https://github.com/geraldmc/irilab2026.git@main --quiet
!pip install --upgrade --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git
#!pip install --upgrade git+https://github.com/geraldmc/irilab2026.git@main --quiet
#!pip install git+https://github.com/geraldmc/irilab2026.git@main --quiet

In [ ]:
# Scientific Python stack.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# The iRI Lab library — pathing, dataset access, and helpers.
import irilab2026 as iri
print(f"irilab2026 version: {iri.__version__}")

In [ ]:
# Mount Drive. We read core_genes.parquet from Notebook 2's outputs in r1_q1/
# and write artifacts from this notebook to the same directory.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Confirm Notebook 2's output is where this notebook expects to find it.
# Catching this here means a path or sync issue surfaces now instead of
# four sections in.
core_path = iri.output_dir("r1_q1") / "core_genes.parquet"
print(f"Looking for: {core_path}")
print(f"Exists:      {core_path.exists()}")
if core_path.exists():
    print(f"Size:        {core_path.stat().st_size / 1024:.1f} KB")

### 1.3) Inputs this notebook needs

Two files:

1. **`core_genes.parquet`** — produced by Notebook 2, sitting in your Drive at `irilab2026_outputs/r1_q1/`. The cell above confirms it is accessible.

2. **Hakkak & Tohidfar 2026 Supplementary File 1** — the meta-analytic DEG list from the published paper. We load this in Section 4. Download instructions and the file naming wrinkle are in that section.